In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')


print("=== ЭТАП 1: ПОДГОТОВКА ПРИЗНАКОВ С ОТБОРОМ ===")

# Загрузка данных
df = pd.read_csv('/kaggle/input/vreros-dataset-a/train.tsv', sep='\t')
df = df[df['target'] > 0].reset_index(drop=True)  # Убираем нулевые оценки

# 1.1 ГЕОГРАФИЧЕСКИЕ ФИЧИ
print("1.1. Создание географических признаков...")
# Парсинг координат
df['coordinates'] = df['coordinates'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df['longitude'] = df['coordinates'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 2 else np.nan)
df['latitude'] = df['coordinates'].apply(lambda x: x[1] if isinstance(x, list) and len(x) == 2 else np.nan)

# Расстояние до центра Москвы
MOSCOW_CENTER = (55.7558, 37.6176)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df['distance_to_center'] = df.apply(
    lambda row: haversine_distance(row['latitude'], row['longitude'], 
                                 MOSCOW_CENTER[0], MOSCOW_CENTER[1])
    if pd.notna(row['latitude']) and pd.notna(row['longitude']) else 20.0, 
    axis=1
)

# Квадраты координат и их произведение
df['lat_sq'] = df['latitude'] ** 2
df['lon_sq'] = df['longitude'] ** 2
df['lat_lon_product'] = df['latitude'] * df['longitude']

# 1.2 КЛАСТЕРИЗАЦИЯ
print("1.2. Кластеризация...")
# Географическая кластеризация
coords = df[['latitude', 'longitude']].dropna()
geo_kmeans = KMeans(n_clusters=12, random_state=42, n_init=10, init='k-means++').fit(coords)
df['geo_cluster'] = np.nan
df.loc[coords.index, 'geo_cluster'] = geo_kmeans.predict(coords)
df['geo_cluster'] = df['geo_cluster'].fillna(-1).astype(int)

# Кластеризация по признакам - используем только 1000м признаки
all_numeric_features = [col for col in df.columns if 
                       col not in ['id', 'name', 'address', 'coordinates', 'target'] and 
                       df[col].dtype in ['float64', 'int64']]

# Оставляем только признаки 1000м радиуса
radius_1000m_features = [f for f in all_numeric_features if '_1000m' in f]
additional_features = ['distance_to_center', 'latitude', 'longitude', 'lat_sq', 'lon_sq', 'lat_lon_product']

features_for_clustering = radius_1000m_features + additional_features

# Стандартизируем и кластеризуем
scaler = StandardScaler()
X_cluster = scaler.fit_transform(df[features_for_clustering].fillna(0))
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10, init='k-means++').fit(X_cluster)
df['feature_cluster'] = kmeans.predict(X_cluster)

# Сохраняем модели
joblib.dump(geo_kmeans, 'geo_kmeans.pkl')
joblib.dump(kmeans, 'feature_kmeans.pkl')
joblib.dump(scaler, 'cluster_scaler.pkl')

# 1.3 ТЕКСТОВЫЕ ФИЧИ
print("1.3. Обработка текстовых данных...")

# Загружаем отзывы
reviews = pd.read_csv('/kaggle/input/vreros-dataset-a/reviews.txv/reviews.tsv', sep='\t')

# 1.3.1 Базовые текстовые признаки
review_stats = reviews.groupby('id').agg(
    review_count=('text', 'count'),
    avg_review_length=('text', lambda x: x.str.len().mean())
).reset_index()

# 1.3.2 TF-IDF с уменьшением размерности
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s]', '', text)
    return text

reviews['clean_text'] = reviews['text'].apply(clean_text)
reviews['text_length'] = reviews['text'].str.len()

# Группируем отзывы по заведению
text_grouped = reviews.groupby('id')['clean_text'].apply(lambda x: ' '.join(x)).reset_index()

# TF-IDF векторизация
tfidf = TfidfVectorizer(max_features=500, min_df=0.01, max_df=0.95)
tfidf_matrix = tfidf.fit_transform(text_grouped['clean_text'])

# Снижение размерности с помощью SVD
svd = TruncatedSVD(n_components=30, random_state=42)
text_features = svd.fit_transform(tfidf_matrix)

# Создаем DataFrame с текстовыми признаками
text_feature_names = [f'text_feature_{i}' for i in range(text_features.shape[1])]
text_features_df = pd.DataFrame(text_features, columns=text_feature_names)
text_features_df['id'] = text_grouped['id']

# 1.3.3 Анализ тональности
sia = SentimentIntensityAnalyzer()
def get_sentiment(text):
    if not isinstance(text, str) or len(text) < 10:
        return 0.0
    return sia.polarity_scores(text)['compound']

# Вычисляем среднюю тональность для каждого заведения
sentiment = reviews.groupby('id')['text'].apply(
    lambda x: x.apply(get_sentiment).mean()
).reset_index(name='avg_sentiment')

# 1.3.4 Собираем все текстовые признаки
text_features = pd.merge(text_features_df, review_stats, on='id', how='left')
text_features = pd.merge(text_features, sentiment, on='id', how='left')

# Заполняем пропуски нулями для заведений без отзывов
text_features = text_features.fillna(0)

# Сохраняем текстовые признаки
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(svd, 'svd_model.pkl')
text_features.to_csv('text_features.csv', index=False)

# 1.4 АГГРЕГАЦИЯ СТАТИСТИК ПО КЛАСТЕРАМ
print("1.4. Агрегация статистик по кластерам...")
# Статистика по гео-кластерам
geo_cluster_stats = df.groupby('geo_cluster').agg({
    'target': ['mean', 'std', 'count', 'median']
}).round(4)
geo_cluster_stats.columns = ['geo_mean', 'geo_std', 'geo_count', 'geo_median']
geo_cluster_stats = geo_cluster_stats.reset_index()

# Статистика по кластерам признаков
feature_cluster_stats = df.groupby('feature_cluster').agg({
    'target': ['mean', 'std', 'count', 'median']
}).round(4)
feature_cluster_stats.columns = ['feature_mean', 'feature_std', 'feature_count', 'feature_median']
feature_cluster_stats = feature_cluster_stats.reset_index()

# Сохраняем статистики
joblib.dump(geo_cluster_stats, 'geo_cluster_stats.pkl')
joblib.dump(feature_cluster_stats, 'feature_cluster_stats.pkl')

# 1.5 ДОБАВЛЕНИЕ ВСЕХ ПРИЗНАКОВ В ОСНОВНОЙ ДАТАФРЕЙМ
print("1.5. Формирование финального набора признаков...")
# Добавляем текстовые признаки
df = pd.merge(df, text_features, on='id', how='left')

# Добавляем статистику по кластерам
df = pd.merge(df, geo_cluster_stats, on='geo_cluster', how='left')
df = pd.merge(df, feature_cluster_stats, on='feature_cluster', how='left')

# Создаем признаки на основе статистики
df['geo_cluster_popularity'] = df['geo_count'] / df['geo_count'].max()
df['feature_cluster_popularity'] = df['feature_count'] / df['feature_count'].max()

# Заполняем пропуски
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())

print("=== ЭТАП 2: ОТБОР ПРИЗНАКОВ ПО КОРРЕЛЯЦИИ ===")

# 2.1 ФУНКЦИЯ ОТБОРА ПРИЗНАКОВ
def select_features_by_correlation(df, target_col, correlation_threshold=0.9):
    """
    Отбирает признаки, удаляя сильно коррелирующие между собой.
    Из группы коррелирующих признаков оставляет тот, который сильнее всего коррелирует с целевой переменной.
    """
    # Выбираем только числовые признаки
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [f for f in numeric_features if f != target_col]
    
    # ИСКЛЮЧАЕМ ПРИЗНАКИ С ID И ДОБАВЛЕННЫЕ ОСОБЫЕ ПРИЗНАКИ
    excluded_features = ['id', 'id.1']  # Явно исключаем ID колонки
    special_features = ['review_count', 'avg_review_length']  # Особые признаки, которые не должны удаляться
    
    numeric_features = [f for f in numeric_features if f not in excluded_features]
    
    # Вычисляем корреляционную матрицу
    corr_matrix = df[numeric_features + [target_col]].corr().abs()
    
    # Верхний треугольник матрицы корреляции
    upper_triangle = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
    
    # Находим пары признаков с корреляцией выше порога
    high_corr_pairs = []
    for col in upper_triangle.columns:
        high_corr = upper_triangle[col][upper_triangle[col] > correlation_threshold]
        for row in high_corr.index:
            high_corr_pairs.append((col, row, high_corr[row]))
    
    # Группируем сильно коррелирующие признаки
    feature_groups = {}
    for feat1, feat2, corr_value in high_corr_pairs:
        group_found = False
        for group in feature_groups.values():
            if feat1 in group or feat2 in group:
                group.update([feat1, feat2])
                group_found = True
                break
        if not group_found:
            feature_groups[len(feature_groups)] = {feat1, feat2}
    
    # Для каждой группы оставляем признак с наибольшей корреляцией с целевой переменной
    features_to_keep = set()
    features_to_remove = set()
    
    for group in feature_groups.values():
        # Находим признак с максимальной корреляцией с таргетом
        best_feature = None
        best_corr = -1
        
        for feature in group:
            if feature in special_features:  # Пропускаем особые признаки
                continue
            corr_with_target = corr_matrix.loc[feature, target_col]
            if corr_with_target > best_corr:
                best_corr = corr_with_target
                best_feature = feature
        
        if best_feature:
            features_to_keep.add(best_feature)
            # Добавляем остальные признаки группы в список на удаление
            features_to_remove.update([f for f in group if f != best_feature])
    
    # Все признаки, которые не попали в группы сильно коррелирующих, тоже оставляем
    all_features_set = set(numeric_features)
    independent_features = all_features_set - set().union(*feature_groups.values())
    features_to_keep.update(independent_features)
    
    # Удаляем признаки, которые должны быть удалены, но сохраняем особые признаки
    features_to_remove = features_to_remove - set(special_features)
    final_features = list(features_to_keep - features_to_remove)
    
    # ДОБАВЛЯЕМ ОСОБЫЕ ПРИЗНАКИ, КОТОРЫЕ ДОЛЖНЫ БЫТЬ В ЛЮБОМ СЛУЧАЕ
    for special_feat in special_features:
        if special_feat in df.columns and special_feat not in final_features:
            final_features.append(special_feat)
    
    print(f"Исходное количество признаков: {len(numeric_features)}")
    print(f"Найдено групп сильно коррелирующих признаков: {len(feature_groups)}")
    print(f"Признаков для удаления: {len(features_to_remove)}")
    print(f"Финальное количество признаков: {len(final_features)}")
    
    return final_features

# 2.2 ВЫПОЛНЯЕМ ОТБОР ПРИЗНАКОВ
print("2.2. Выполнение отбора признаков...")
selected_features = select_features_by_correlation(df, 'target', correlation_threshold=0.9)

# Добавляем категориальные признаки (они не числовые, поэтому не попали в отбор)
categorical_features = ['category', 'geo_cluster', 'feature_cluster', 'name', 'address']
selected_features.extend([f for f in categorical_features if f in df.columns])

# Удаляем дубликаты и исключаем ID колонки
selected_features = list(set(selected_features))
selected_features = [f for f in selected_features if 'id' not in f.lower()]

print(f"Общее количество признаков после отбора: {len(selected_features)}")

# Сохраняем отобранные признаки
joblib.dump(selected_features, 'selected_features.pkl')

# 2.3 ФИЛЬТРУЕМ ДАННЫЕ
# УБИРАЕМ ВСЕ КОЛОНКИ С ID ИЗ ФИНАЛЬНОГО ДАТАФРЕЙМА
id_columns = [col for col in df.columns if 'id' in col.lower()]
features_without_id = [f for f in selected_features if f not in id_columns]

df_filtered = df[['target'] + features_without_id].copy()

# Очищаем названия столбцов от запрещенных символов для XGBoost
def clean_column_names(df):
    df_clean = df.copy()
    df_clean.columns = [col.replace('[', '_').replace(']', '_').replace('<', '_lt_').replace('>', '_gt_') 
                       for col in df_clean.columns]
    return df_clean

df_filtered = clean_column_names(df_filtered)

# Обновляем список признаков с очищенными названиями
selected_features_clean = [col.replace('[', '_').replace(']', '_').replace('<', '_lt_').replace('>', '_gt_') 
                          for col in features_without_id]

# Сохраняем обработанный датафрейм
df_filtered.to_csv('processed_train_filtered.csv', index=False)

print("✅ Обработка и отбор признаков завершены!")
print(f"Финальные признаки: {list(df_filtered.columns)}")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


=== ЭТАП 1: ПОДГОТОВКА ПРИЗНАКОВ С ОТБОРОМ ===
1.1. Создание географических признаков...
1.2. Кластеризация...
1.3. Обработка текстовых данных...
1.4. Агрегация статистик по кластерам...
1.5. Формирование финального набора признаков...
=== ЭТАП 2: ОТБОР ПРИЗНАКОВ ПО КОРРЕЛЯЦИИ ===
2.2. Выполнение отбора признаков...
Исходное количество признаков: 331
Найдено групп сильно коррелирующих признаков: 9
Признаков для удаления: 275
Финальное количество признаков: 56
Общее количество признаков после отбора: 59
✅ Обработка и отбор признаков завершены!
Финальные признаки: ['target', 'geo_count', 'text_feature_21', 'text_feature_4', 'text_feature_28', 'text_feature_7', 'avg_review_length', 'childrens_websites_1000m', 'text_feature_2', 'text_feature_8', 'text_feature_22', 'mean_income_300m', 'geo_mean', 'text_feature_1', 'lat_sq', 'goods_for_moms_and_babies_300m', 'childrens_sports_1000m', 'childrens_sports_300m', 'text_feature_10', 'geo_std', 'text_feature_17', 'mean_income_1000m', 'address', 'te

In [2]:
print("=== ЭТАП 3: ОБУЧЕНИЕ CATBOOST С КРОСС-ВАЛИДАЦИЕЙ ===")

# 3.1 ЗАГРУЗКА ОБРАБОТАННЫХ ДАННЫХ
print("3.1. Загрузка отфильтрованных данных...")
df = pd.read_csv('processed_train_filtered.csv')
df = df.fillna("NAN")

# Определяем целевую переменную и признаки
target = 'target'
all_features = [col for col in df.columns if col not in ['id', target, 'coordinates', 'id.1']]

# Подготовка данных
X = df[all_features]
y = df[target]

print(f"Общий размер данных: {X.shape[0]} записей, {X.shape[1]} признаков")

# # 3.2 КРОСС-ВАЛИДАЦИЯ (5 ФОЛДОВ)
# print("\n3.2. Проведение 5-фолдовой кросс-валидации...")

# from sklearn.model_selection import KFold

# kf = KFold(n_splits=5, shuffle=True, random_state=42)
# mae_scores = []

cat_indices = [X.columns.get_loc(col) for col in categorical_features if col in X.columns]

cat_params = {
    'iterations': 5200,
    'learning_rate': 0.01,
    'depth': 8,
    'l2_leaf_reg': 4,
    'loss_function': 'MAE',
    'cat_features': cat_indices,
    'early_stopping_rounds': 150,
    'verbose': 100,
    'random_seed': 42
}

# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f"\n--- Fold {fold+1} из 5 ---")
    
#     # Разделение данных
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
#     # Обучение модели
#     model = CatBoostRegressor(**cat_params)
#     model.fit(
#         X_train, y_train,
#         eval_set=(X_val, y_val),
#         use_best_model=True,
#         verbose=200
#     )
    
#     # Оценка качества
#     val_pred = model.predict(X_val)
#     fold_mae = mean_absolute_error(y_val, val_pred)
#     mae_scores.append(fold_mae)
#     print(f"MAE на валидации: {fold_mae:.4f}")

# # Итоговая оценка
# average_mae = np.mean(mae_scores)
# print("\n" + "="*50)
# print(f"Средняя MAE по 5 фолдам: {average_mae:.4f}")
# print(f"Стандартное отклонение: {np.std(mae_scores):.4f}")
# print("="*50)

# 3.3 ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ НА ВСЕХ ДАННЫХ
print("\n3.3. Обучение финальной модели на всём датасете...")

final_model = CatBoostRegressor(**cat_params)
final_model.fit(
    X, y,
    verbose=100
)

# Оценка на случайной валидационной выборке (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
val_pred = final_model.predict(X_val)
final_mae = mean_absolute_error(y_val, val_pred)
print(f"\nMAE финальной модели на валидации: {final_mae:.4f}")
print(f"Ожидаемый скор: {1 / (1 + final_mae):.4f}")

# Сохраняем только финальную модель
final_model.save_model('catboost_model.cbm')
print("\n✅ CatBoost обучена и сохранена!")

=== ЭТАП 3: ОБУЧЕНИЕ CATBOOST С КРОСС-ВАЛИДАЦИЕЙ ===
3.1. Загрузка отфильтрованных данных...
Общий размер данных: 37167 записей, 59 признаков

3.3. Обучение финальной модели на всём датасете...
0:	learn: 0.3307959	total: 201ms	remaining: 17m 24s
100:	learn: 0.2792945	total: 7.05s	remaining: 5m 55s
200:	learn: 0.2590550	total: 13.8s	remaining: 5m 44s
300:	learn: 0.2517079	total: 21s	remaining: 5m 41s
400:	learn: 0.2469396	total: 28.3s	remaining: 5m 38s
500:	learn: 0.2434193	total: 35.8s	remaining: 5m 35s
600:	learn: 0.2405998	total: 43.2s	remaining: 5m 30s
700:	learn: 0.2383561	total: 50.7s	remaining: 5m 25s
800:	learn: 0.2363150	total: 58.3s	remaining: 5m 20s
900:	learn: 0.2345045	total: 1m 5s	remaining: 5m 14s
1000:	learn: 0.2326766	total: 1m 13s	remaining: 5m 7s
1100:	learn: 0.2308714	total: 1m 20s	remaining: 5m
1200:	learn: 0.2292385	total: 1m 28s	remaining: 4m 53s
1300:	learn: 0.2275437	total: 1m 35s	remaining: 4m 47s
1400:	learn: 0.2259963	total: 1m 43s	remaining: 4m 40s
1500:	lea

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=== ЭТАП 4: ПРЕДСКАЗАНИЕ НА ТЕСТОВЫХ ДАННЫХ ===")

# 4.1 ЗАГРУЗКА ТЕСТОВЫХ ДАННЫХ
print("4.1. Загрузка тестовых данных...")
test_df = pd.read_csv('/kaggle/input/vreros-dataset-a/test.tsv', sep='\t')

# 4.2 ПРИМЕНЕНИЕ ПРЕОБРАЗОВАНИЙ К ТЕСТОВЫМ ДАННЫМ
print("4.2. Применение преобразований...")

# 4.2.1 ГЕОГРАФИЧЕСКИЕ ПРИЗНАКИ
print("  - Географические признаки...")
test_df['coordinates'] = test_df['coordinates'].apply(lambda x: eval(x) if isinstance(x, str) else x)
test_df['longitude'] = test_df['coordinates'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 2 else np.nan)
test_df['latitude'] = test_df['coordinates'].apply(lambda x: x[1] if isinstance(x, list) and len(x) == 2 else np.nan)

# Расстояние до центра Москвы
MOSCOW_CENTER = (55.7558, 37.6176)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

test_df['distance_to_center'] = test_df.apply(
    lambda row: haversine_distance(row['latitude'], row['longitude'], 
                                 MOSCOW_CENTER[0], MOSCOW_CENTER[1])
    if pd.notna(row['latitude']) and pd.notna(row['longitude']) else 20.0, 
    axis=1
)

# Квадраты координат и их произведение
test_df['lat_sq'] = test_df['latitude'] ** 2
test_df['lon_sq'] = test_df['longitude'] ** 2
test_df['lat_lon_product'] = test_df['latitude'] * test_df['longitude']

# 4.2.2 КЛАСТЕРИЗАЦИЯ
print("  - Кластеризация...")

# Загружаем сохраненные модели кластеризации
geo_kmeans = joblib.load('geo_kmeans.pkl')
feature_kmeans = joblib.load('feature_kmeans.pkl')
cluster_scaler = joblib.load('cluster_scaler.pkl')

# Географическая кластеризация
coords_test = test_df[['latitude', 'longitude']].dropna()
test_df['geo_cluster'] = -1
if len(coords_test) > 0:
    test_df.loc[coords_test.index, 'geo_cluster'] = geo_kmeans.predict(coords_test)

# Кластеризация по признакам - используем только 1000м признаки
radius_1000m_features = [f for f in test_df.columns if '_1000m' in f]
additional_features = ['distance_to_center', 'latitude', 'longitude', 'lat_sq', 'lon_sq', 'lat_lon_product']
features_for_clustering = radius_1000m_features + additional_features

# Заполняем пропуски и стандартизируем
X_cluster_test = cluster_scaler.transform(test_df[features_for_clustering].fillna(0))
test_df['feature_cluster'] = feature_kmeans.predict(X_cluster_test)

# 4.2.3 ТЕКСТОВЫЕ ПРИЗНАКИ
print("  - Текстовые признаки...")

# Загружаем отзывы для тестовых данных
try:
    reviews_test = pd.read_csv('/kaggle/input/vreros-dataset-a/reviews.txv/reviews.tsv', sep='\t')
    # Фильтруем отзывы только для тестовых id
    test_ids = set(test_df['id'])
    reviews_test = reviews_test[reviews_test['id'].isin(test_ids)]
    
    # Базовые текстовые признаки
    review_stats_test = reviews_test.groupby('id').agg(
        review_count=('text', 'count'),
        avg_review_length=('text', lambda x: x.str.len().mean())
    ).reset_index()

    # TF-IDF преобразование
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s]', '', text)
        return text

    reviews_test['clean_text'] = reviews_test['text'].apply(clean_text)
    
    # Группируем отзывы по заведению
    text_grouped_test = reviews_test.groupby('id')['clean_text'].apply(lambda x: ' '.join(x)).reset_index()
    
    # Загружаем сохраненные TF-IDF и SVD модели
    tfidf = joblib.load('tfidf_vectorizer.pkl')
    svd = joblib.load('svd_model.pkl')
    
    # Преобразуем текст
    tfidf_matrix_test = tfidf.transform(text_grouped_test['clean_text'])
    text_features_test = svd.transform(tfidf_matrix_test)
    
    # Создаем DataFrame с текстовыми признаками
    text_feature_names = [f'text_feature_{i}' for i in range(text_features_test.shape[1])]
    text_features_df_test = pd.DataFrame(text_features_test, columns=text_feature_names)
    text_features_df_test['id'] = text_grouped_test['id']
    
    # Анализ тональности
    sia = SentimentIntensityAnalyzer()
    def get_sentiment(text):
        if not isinstance(text, str) or len(text) < 10:
            return 0.0
        return sia.polarity_scores(text)['compound']

    sentiment_test = reviews_test.groupby('id')['text'].apply(
        lambda x: x.apply(get_sentiment).mean()
    ).reset_index(name='avg_sentiment')
    
    # Собираем все текстовые признаки
    text_features_test = pd.merge(text_features_df_test, review_stats_test, on='id', how='left')
    text_features_test = pd.merge(text_features_test, sentiment_test, on='id', how='left')
    text_features_test = text_features_test.fillna(0)
    
    # Добавляем текстовые признаки к тестовым данным
    test_df = pd.merge(test_df, text_features_test, on='id', how='left')
    
except Exception as e:
    print(f"    Предупреждение: не удалось обработать отзывы для теста: {e}")
    # Создаем пустые текстовые признаки
    text_feature_names = [f'text_feature_{i}' for i in range(30)]
    for col in text_feature_names + ['review_count', 'avg_review_length', 'avg_sentiment']:
        test_df[col] = 0

# 4.2.4 СТАТИСТИКИ ПО КЛАСТЕРАМ
print("  - Статистики по кластерам...")

# Загружаем сохраненные статистики
geo_cluster_stats = joblib.load('geo_cluster_stats.pkl')
feature_cluster_stats = joblib.load('feature_cluster_stats.pkl')

# Добавляем статистику по кластерам
test_df = pd.merge(test_df, geo_cluster_stats, on='geo_cluster', how='left')
test_df = pd.merge(test_df, feature_cluster_stats, on='feature_cluster', how='left')

# Создаем признаки на основе статистики
test_df['geo_cluster_popularity'] = test_df['geo_count'] / test_df['geo_count'].max() if test_df['geo_count'].max() > 0 else 0
test_df['feature_cluster_popularity'] = test_df['feature_count'] / test_df['feature_count'].max() if test_df['feature_count'].max() > 0 else 0

# 4.3 ПОДГОТОВКА ПРИЗНАКОВ ДЛЯ ПРЕДСКАЗАНИЯ
print("4.3. Подготовка признаков для предсказания...")

# Загружаем список отобранных признаков
selected_features = joblib.load('selected_features.pkl')

# Очищаем названия столбцов от запрещенных символов
def clean_column_names(df):
    df_clean = df.copy()
    df_clean.columns = [col.replace('[', '_').replace(']', '_').replace('<', '_lt_').replace('>', '_gt_') 
                       for col in df_clean.columns]
    return df_clean

test_df = clean_column_names(test_df)

# Обновляем список признаков с очищенными названиями
selected_features_clean = [col.replace('[', '_').replace(']', '_').replace('<', '_lt_').replace('>', '_gt_') 
                          for col in selected_features]

# Выбираем только нужные признаки
available_features = [f for f in selected_features_clean if f in test_df.columns]
missing_features = [f for f in selected_features_clean if f not in test_df.columns]

if missing_features:
    print(f"  Предупреждение: отсутствуют признаки: {missing_features}")
    # Добавляем отсутствующие признаки с нулевыми значениями
    for feature in missing_features:
        test_df[feature] = 0

# Фильтруем тестовые данные по отобранным признакам
X_test = test_df[selected_features_clean].copy()

# Заполняем пропуски
for col in X_test.columns:
    if X_test[col].dtype in ['float64', 'int64']:
        X_test[col] = X_test[col].fillna(X_test[col].median())
    else:
        X_test[col] = X_test[col].fillna('unknown')

print(f"  Используется {len(selected_features_clean)} признаков для предсказания")

=== ЭТАП 4: ПРЕДСКАЗАНИЕ НА ТЕСТОВЫХ ДАННЫХ ===
4.1. Загрузка тестовых данных...


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


4.2. Применение преобразований...
  - Географические признаки...
  - Кластеризация...
  - Текстовые признаки...
  - Статистики по кластерам...
4.3. Подготовка признаков для предсказания...
  Используется 59 признаков для предсказания


In [4]:
# 4.5 ЗАГРУЗКА МОДЕЛИ И ПРЕДСКАЗАНИЕ
print("4.5. Загрузка модели CatBoost и предсказание...")

# Загружаем модель
cat_model = CatBoostRegressor()
cat_model.load_model('catboost_model.cbm')

# Предсказание
final_predictions = cat_model.predict(X_test)

# Обеспечиваем, что предсказания в допустимом диапазоне
final_predictions = np.clip(final_predictions, 1, 5)

# 4.8 СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
print("4.8. Сохранение результатов...")

# Создаем DataFrame с результатами
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': final_predictions
})

# Сохраняем предсказания
submission.to_csv('submission.csv', index=False)

print("✅ Предсказания завершены!")
print(f"Предсказано значений: {len(submission)}")
print(f"Диапазон предсказаний: {final_predictions.min():.3f} - {final_predictions.max():.3f}")
print(f"Среднее предсказание: {final_predictions.mean():.3f}")

4.5. Загрузка модели CatBoost и предсказание...
4.8. Сохранение результатов...
✅ Предсказания завершены!
Предсказано значений: 9276
Диапазон предсказаний: 2.581 - 4.843
Среднее предсказание: 3.778
